## Module 1: Data Ingestion & Preprocessing
> This module handles all data loading, text tokenization, vocabulary mapping, and sequence preparation. It reads raw text data from a CSV file, constructs a frequency-based vocabulary dictionary with reserved padding and unknown tokens, pads or truncates text sequences to a uniform length, and splits the dataset into training (70%), validation (15%), and testing (15%) subsets for model training.


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

def build_vocab(texts, max_vocab_size):
    """Builds a vocabulary mapping words to unique integer indices."""
    words = [word.lower() for text in texts for word in str(text).split()]
    word_counts = Counter(words)

    # Reserve index 0 for padding (<PAD>) and 1 for unknown tokens (<UNK>)
    most_common = word_counts.most_common(max_vocab_size - 2)
    vocab = {word: idx + 2 for idx, (word, _) in enumerate(most_common)}
    vocab["<PAD>"] = 0
    vocab["<UNK>"] = 1
    return vocab

def text_to_sequence(text, vocab, max_len):
    """Converts a text string into a fixed-length sequence of integer tokens."""
    tokens = str(text).lower().split()
    seq = [vocab.get(token, 1) for token in tokens]

    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))   # Pad with 0
    else:
        seq = seq[:max_len]                 # Truncate
    return seq

def load_and_preprocess_data(csv_filepath, max_len=20, max_vocab_size=1000, random_state=42):
    """Loads CSV, builds vocabulary, encodes texts, and splits into train/val/test sets."""
    df = pd.read_csv(csv_filepath)
    vocab = build_vocab(df["text"].tolist(), max_vocab_size)
    encoded_texts = [text_to_sequence(txt, vocab, max_len) for txt in df["text"]]

    X_data = np.array(encoded_texts)
    y_data = np.array(df["label"].values)
    num_classes = len(np.unique(y_data))

    # Partitioning (70% Train, 15% Validation, 15% Test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X_data, y_data, test_size=0.30, random_state=random_state
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=random_state
    )

    return (X_train, y_train), (X_val, y_val), (X_test, y_test), vocab, num_classes

## Module 2: Configuration Management
> This module centralizes all model hyperparameter definitions, random seeds, file paths, and metadata into a single dictionary structure. Centralizing these values keeps the pipeline reproducible, manageable, and easy to adjust during hyperparameter tuning without changing underlying execution code.

In [ ]:
import numpy as np
import tensorflow as tf

def set_reproducibility_seed(seed=42):
    """Sets random seeds for NumPy and TensorFlow."""
    np.random.seed(seed)
    tf.random.set_seed(seed)

CONFIG = {
    "run_id": "EXP-01-KERAS-BASELINE",
    "csv_filepath": "/content/drive/MyDrive/Deep Learning/Lab Exercise PT-P2/sample_dataset 1.csv",
    "max_len": 20,
    "max_vocab_size": 1000,
    "embed_dim": 32,
    "hidden_dim": 64,
    "learning_rate": 0.005,
    "batch_size": 32,
    "epochs": 15,
    "optimizer_type": "Adam",
    "dropout_rate": 0.1
}

## Module 3: Neural Network Architecture
> This module constructs the neural network using the Keras Sequential API. It maps integer word tokens into dense embeddings, pools sequence vectors via global average pooling, passes feature representations through a fully connected dense layer with ReLU activation and dropout regularization, and outputs class probability distributions using a softmax output layer.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense, Dropout

def build_model(vocab_size, max_len, num_classes, config):
    """Constructs the Keras Sequential text classification model."""
    model = Sequential()

    # Embedding Layer
    model.add(
        Embedding(
            input_dim=vocab_size,
            output_dim=config["embed_dim"],
            input_length=max_len,
            mask_zero=True
        )
    )

    # Pooling & Dense Layers
    model.add(GlobalAveragePooling1D())
    model.add(Dense(config["hidden_dim"], activation="relu"))
    model.add(Dropout(config["dropout_rate"]))
    model.add(Dense(num_classes, activation="softmax"))

    return model

## Module 4: Optimizer & Compilation
> This module configures the learning strategy and loss function for the model. Based on configuration parameters, it initializes either an Adam or SGD optimizer with a specified learning rate and compiles the neural network using sparse categorical cross-entropy loss and accuracy metrics.

In [ ]:
from tensorflow.keras.optimizers import Adam, SGD

def compile_model(model, config):
    """Instantiates the specified optimizer and compiles the Keras model."""
    if config["optimizer_type"] == "Adam":
        opt = Adam(learning_rate=config["learning_rate"])
    else:
        opt = SGD(learning_rate=config["learning_rate"])

    model.compile(
        loss="sparse_categorical_crossentropy",
        optimizer=opt,
        metrics=["accuracy"]
    )
    return model

## Module 5: Operational Training & Evaluation Loop
> This module manages model execution by fitting the neural network on training samples while validating performance across training epochs. Post-training, it evaluates model generalization by generating prediction classes on validation data, calculating weighted F1-scores, and printing structured logs for experiment tracking.

In [ ]:
import numpy as np
from sklearn.metrics import f1_score

def train_and_evaluate(model, train_data, val_data, config):
    """Fits the model on training data, evaluates on validation data, and logs metrics."""
    X_train, y_train = train_data
    X_val, y_val = val_data

    print(f"--- Starting Training Run: {config['run_id']} ---")

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        verbose=1
    )

    # Predictions & F1 Evaluation
    val_preds_prob = model.predict(X_val)
    val_preds = np.argmax(val_preds_prob, axis=1)

    val_f1 = f1_score(y_val, val_preds, average="weighted")
    val_loss = history.history["val_loss"][-1]

    # Structured Logging Output
    print("\n=== EXP LOG ENTRY ===")
    print(
        f"ID: {config['run_id']} | LR: {config['learning_rate']} | "
        f"Hidden: {config['hidden_dim']} | Dropout: {config['dropout_rate']} | "
        f"Final Val Loss: {val_loss:.4f} | Final Val F1: {val_f1:.4f}"
    )

    return history

## Module 6: Performance Visualization
> This module renders diagnostic charts to visually analyze learning behavior across training epochs using Matplotlib. By plotting training and validation loss trajectories together, it provides visual indicators of model convergence, underfitting, or overfitting.

In [ ]:
import matplotlib.pyplot as plt

def plot_loss_curves(history, config):
    """Plots training and validation loss curves over training epochs."""
    train_losses = history.history["loss"]
    val_losses = history.history["val_loss"]
    epochs_range = range(1, config["epochs"] + 1)

    plt.figure(figsize=(8, 5))
    plt.plot(
        epochs_range, train_losses, label="Training Loss",
        color="blue", linewidth=2, marker="o"
    )
    plt.plot(
        epochs_range, val_losses, label="Validation Loss",
        color="red", linewidth=2, linestyle="--", marker="s"
    )

    plt.title(
        f"Performance Monitoring: Loss Curves (CONFIG={config['run_id']})",
        fontsize=12, fontweight="bold"
    )
    plt.xlabel("Epochs", fontsize=10)
    plt.ylabel("Loss (Sparse Categorical Cross-Entropy)", fontsize=10)
    plt.xticks(epochs_range)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

## Main Pipeline Execution

In [ ]:
from google.colab import drive
# 0. Mount Drive
drive.mount('/content/drive')

# 1. Reproducibility & Configuration
set_reproducibility_seed(42)

# 2. Data Loading
train_data, val_data, test_data, vocab, num_classes = load_and_preprocess_data(
    csv_filepath=CONFIG["csv_filepath"],
    max_len=CONFIG["max_len"],
    max_vocab_size=CONFIG["max_vocab_size"]
)

# 3. Model Building & Compilation
model = build_model(
    vocab_size=len(vocab),
    max_len=CONFIG["max_len"],
    num_classes=num_classes,
    config=CONFIG
)
model = compile_model(model, CONFIG)
model.summary()

# 4. Fit & Evaluate
history = train_and_evaluate(model, train_data, val_data, CONFIG)

# 5. Visualize Results
plot_loss_curves(history, CONFIG)